<a href="https://colab.research.google.com/github/Rajapaksha-RMDD/pusl3123-ai-ml-coursework/blob/main/Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 – MLP Classifier on CIFAR-10


In [ ]:
import json, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("Device:", device, "-", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (switch runtime to GPU)")

SEED, EPOCHS, BATCH_SIZE, LR = 42, 10, 64, 1e-3

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

##cache the dataset on Google Drive


In [ ]:
USE_DRIVE = False
DATA_ROOT = "./data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/cifar10_data"
print("Data root:", DATA_ROOT)

##Data loading, preprocessing and partitions

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

train_full = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=transform)

train_dataset, val_dataset = random_split(
    train_full, [45000, 5000], generator=torch.Generator().manual_seed(42)   # identical to Exercise 2
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
# separate non-shuffled loader so final training accuracy is measured the same way as val/test
train_eval_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = train_full.classes
print("Train:", len(train_dataset), "| Validation:", len(val_dataset), "| Test:", len(test_dataset))
print("Classes:", classes)

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape, "-> flattened:", images.flatten(1).shape)

In [ ]:
imgs, labs = next(iter(train_loader))
plt.figure(figsize=(10, 5))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow((imgs[i] * 0.5 + 0.5).permute(1, 2, 0).numpy())
    plt.title(classes[labs[i]]); plt.axis("off")
plt.tight_layout(); plt.show()

## Model – MLP with three trainable layers


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3072, 512), nn.ReLU(),
            nn.Linear(512, 256),  nn.ReLU(),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.network(x)

set_seed(SEED)
model = MLP().to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters:", total_params)   # expected 1,707,274

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

##Training and evaluation functions

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total

def train_one_epoch(model, loader):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total

##Train for 10 epochs (single loop, all 10 epochs recorded)

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
t0 = time.time()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader)
    va_loss, va_acc = evaluate(model, val_loader)
    for k, v in zip(history, (tr_loss, va_loss, tr_acc, va_acc)):
        history[k].append(v)
    print(f"Epoch [{epoch+1:2d}/{EPOCHS}] train loss {tr_loss:.4f} acc {tr_acc:.2f}% | "
          f"val loss {va_loss:.4f} acc {va_acc:.2f}%")
print(f"Training time: {time.time()-t0:.0f} s")

##Training and validation loss / accuracy plots

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(ep, history["train_loss"], marker="o", label="Training loss")
ax[0].plot(ep, history["val_loss"],   marker="o", label="Validation loss")
ax[0].set_title("MLP – training and validation loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[1].plot(ep, history["train_acc"], marker="o", label="Training accuracy")
ax[1].plot(ep, history["val_acc"],   marker="o", label="Validation accuracy")
ax[1].set_title("MLP – training and validation accuracy"); ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy (%)")
for a in ax: a.set_xticks(list(ep)); a.grid(True); a.legend()
plt.tight_layout(); plt.savefig("ex1_mlp_curves.png", dpi=150); plt.show()

##Final evaluation

In [ ]:
train_loss_eval, train_acc_eval = evaluate(model, train_eval_loader)
val_loss_f, val_acc_f   = evaluate(model, val_loader)
test_loss_f, test_acc_f = evaluate(model, test_loader)

print("MLP final results")
print("-" * 40)
print(f"Training   accuracy: {train_acc_eval:.2f}%   loss: {train_loss_eval:.4f}")
print(f"Validation accuracy: {val_acc_f:.2f}%   loss: {val_loss_f:.4f}")
print(f"Test       accuracy: {test_acc_f:.2f}%   loss: {test_loss_f:.4f}")
print(f"Trainable parameters: {total_params:,}")
print(f"Train - test gap    : {train_acc_eval - test_acc_f:.2f} percentage points")

In [ ]:
# Per-class accuracy and confusion matrix on the test set
from sklearn.metrics import confusion_matrix
model.eval()
preds, trues = [], []
with torch.no_grad():
    for x, y in test_loader:
        preds.extend(model(x.to(device)).argmax(1).cpu().numpy()); trues.extend(y.numpy())
preds, trues = np.array(preds), np.array(trues)
for i, c in enumerate(classes):
    print(f"{c:12s}: {100 * (preds[trues == i] == i).mean():.2f}%")

cm = confusion_matrix(trues, preds)
plt.figure(figsize=(8, 7)); plt.imshow(cm); plt.colorbar()
plt.xticks(range(10), classes, rotation=45); plt.yticks(range(10), classes)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("MLP confusion matrix (test set)")
plt.tight_layout(); plt.savefig("ex1_mlp_confusion.png", dpi=150); plt.show()

##Save results for the MLP vs CNN comparison in Ex 2

In [ ]:
mlp_results = {
    "train_accuracy": train_acc_eval, "validation_accuracy": val_acc_f, "test_accuracy": test_acc_f,
    "train_loss_final_epoch": history["train_loss"][-1], "validation_loss": val_loss_f, "test_loss": test_loss_f,
    "parameters": total_params,
    "history": history,
}
with open("mlp_results.json", "w") as f:
    json.dump(mlp_results, f, indent=2)
torch.save(model.state_dict(), "mlp_cifar10_model.pth")
print("Saved mlp_results.json and mlp_cifar10_model.pth")


try:
    from google.colab import files
    files.download("mlp_results.json")
except Exception as e:
    print("Download skipped:", e)